# Section 4 — Higher-Order Taylor Methods

[← Euler Error](03_euler_error.ipynb) · [Module 3](README.md)

Euler's method keeps only the first derivative term in the Taylor expansion. Keeping more terms gives higher-order Taylor methods:

$$
y(t+h)=y(t)+h y'(t)+\frac{h^2}{2}y''(t)+\frac{h^3}{6}y^{(3)}(t)+\cdots.
$$

The price is that we must compute higher derivatives of the solution and rewrite them in terms of $t$ and $y$.

### Learning goals

- construct second- and third-order Taylor methods;
- implement both methods;
- compare their accuracy with Euler's method;
- verify the expected convergence order numerically;
- understand why Runge–Kutta methods are attractive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Use the same IVP as before

Consider

$$
y'=t e^{3t}-2y,\qquad y(0)=0.
$$

The exact solution is

$$
y(t)=e^{3t}\left(\frac{t}{5}-\frac{1}{25}\right)+\frac{1}{25}e^{-2t}.
$$

For a Taylor method we need derivatives beyond $y'$.

Starting with

$$
y'=t e^{3t}-2y,
$$

differentiation gives

$$
y''=e^{3t}(1+t)+4y,
$$

and

$$
y^{(3)}=e^{3t}(4+7t)-8y.
$$

In [ ]:
def y1(t, y):
    return t*np.exp(3*t) - 2*y


def y2(t, y):
    return np.exp(3*t)*(1 + t) + 4*y


def y3(t, y):
    return np.exp(3*t)*(4 + 7*t) - 8*y


def exact(t):
    return np.exp(3*t)*(t/5 - 1/25) + np.exp(-2*t)/25

## 2. Taylor methods of order 2 and 3

The second-order update is

$$
w_{n+1}=w_n+h y'(t_n,w_n)+\frac{h^2}{2}y''(t_n,w_n),
$$

while the third-order update adds

$$
\frac{h^3}{6}y^{(3)}(t_n,w_n).
$$

In [ ]:
def taylor2(t0, y0, h, n_steps):
    t = np.empty(n_steps + 1)
    w = np.empty(n_steps + 1)
    t[0], w[0] = t0, y0
    for n in range(n_steps):
        w[n+1] = w[n] + h*y1(t[n], w[n]) + (h**2/2)*y2(t[n], w[n])
        t[n+1] = t[n] + h
    return t, w


def taylor3(t0, y0, h, n_steps):
    t = np.empty(n_steps + 1)
    w = np.empty(n_steps + 1)
    t[0], w[0] = t0, y0
    for n in range(n_steps):
        w[n+1] = (w[n]
                  + h*y1(t[n], w[n])
                  + (h**2/2)*y2(t[n], w[n])
                  + (h**3/6)*y3(t[n], w[n]))
        t[n+1] = t[n] + h
    return t, w


def euler(t0, y0, h, n_steps):
    t = np.empty(n_steps + 1)
    w = np.empty(n_steps + 1)
    t[0], w[0] = t0, y0
    for n in range(n_steps):
        w[n+1] = w[n] + h*y1(t[n], w[n])
        t[n+1] = t[n] + h
    return t, w

## 3. Compare the methods with $h=0.5$

Two steps take us from $t=0$ to $t=1$.

In [ ]:
h = 0.5
n_steps = 2
methods = {
    "Euler": euler,
    "Taylor 2": taylor2,
    "Taylor 3": taylor3,
}

print("method          approximation      exact y(1)        error")
for name, method in methods.items():
    t, w = method(0.0, 0.0, h, n_steps)
    err = abs(exact(1.0) - w[-1])
    print(f"{name:10s}      {w[-1]:12.8f}    {exact(1.0):12.8f}   {err:10.6f}")

In [ ]:
t_fine = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label="exact")

for name, method in methods.items():
    t, w = method(0.0, 0.0, h, n_steps)
    ax.plot(t, w, marker="o", linewidth=1.5, label=name)

ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Euler and higher-order Taylor methods, h=0.5")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 4. Check the convergence order

A method of global order $p$ should satisfy approximately

$$
E(h)\approx C h^p.
$$

So halving $h$ should reduce the error by a factor of approximately $2^p$.

In [ ]:
step_sizes = np.array([0.5, 0.25, 0.125, 0.0625])
results = {"Taylor 2": [], "Taylor 3": []}

for h in step_sizes:
    n_steps = round(1/h)
    for name, method in [("Taylor 2", taylor2), ("Taylor 3", taylor3)]:
        _, w = method(0.0, 0.0, h, n_steps)
        results[name].append(abs(exact(1.0) - w[-1]))

for name in results:
    print()
    print(name)
    print("     h          error       observed p")
    previous = None
    for h, err in zip(step_sizes, results[name]):
        if previous is None:
            p_text = "   --"
        else:
            p = np.log(previous/err)/np.log(2)
            p_text = f"{p:7.4f}"
        print(f"{h:7.4f}   {err:12.6e}   {p_text}")
        previous = err

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, errs in results.items():
    ax.loglog(step_sizes, errs, marker="o", label=name)

ax.set_xlabel("step size h")
ax.set_ylabel("absolute error at t=1")
ax.set_title("Convergence of higher-order Taylor methods")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.show()

## 5. Why not keep going with Taylor methods?

Higher-order Taylor methods are accurate, but each additional order requires another derivative. For a general ODE $y'=f(t,y)$, repeated differentiation introduces partial derivatives such as $f_t$, $f_y$, and increasingly complicated chain-rule terms.

That motivates the Runge–Kutta idea: **obtain higher-order accuracy using only carefully chosen evaluations of $f(t,y)$** rather than explicitly calculating higher derivatives.

That is where Section 5 will begin.

[← Euler Error](03_euler_error.ipynb) · [↑ Module 3](README.md)